# DATA SCRAPPING

- Beatiful Soup
- https://beautiful-soup-4.readthedocs.io/en/latest/

In [1]:
import requests
from bs4 import BeautifulSoup

url = "https://quotes.toscrape.com/"
response = requests.get(url)

soup = BeautifulSoup(response.text, "html.parser")

quotes = soup.find_all("span", class_="text")

for q in quotes:
    print(q.text)

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
“It is our choices, Harry, that show what we truly are, far more than our abilities.”
“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”
“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”
“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”
“Try not to become a man of success. Rather become a man of value.”
“It is better to be hated for what you are than to be loved for what you are not.”
“I have not failed. I've just found 10,000 ways that won't work.”
“A woman is like a tea bag; you never know how strong it is until it's in hot water.”
“A day without sunshine is like, you know, night.”


### Extracting structured data (multiple fields)

In [2]:
import requests
from bs4 import BeautifulSoup

url = "https://quotes.toscrape.com/"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

data = []

for quote in soup.find_all("div", class_="quote"):
    text = quote.find("span", class_="text").text
    author = quote.find("small", class_="author").text
    tags = [tag.text for tag in quote.find_all("a", class_="tag")]

    data.append({
        "text": text,
        "author": author,
        "tags": tags
    })

print(data[:3])

[{'text': '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”', 'author': 'Albert Einstein', 'tags': ['change', 'deep-thoughts', 'thinking', 'world']}, {'text': '“It is our choices, Harry, that show what we truly are, far more than our abilities.”', 'author': 'J.K. Rowling', 'tags': ['abilities', 'choices']}, {'text': '“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”', 'author': 'Albert Einstein', 'tags': ['inspirational', 'life', 'live', 'miracle', 'miracles']}]


### Saving

In [3]:
import pandas as pd

df = pd.DataFrame(data)
df.to_csv("quotes.csv", index=False)

### API scraping

In [5]:
import requests

url = "https://api.github.com/repos/python/cpython"
response = requests.get(url)

data = response.json()

print(data["name"], data["stargazers_count"])

cpython 72003


### From Wikipedia

In [11]:
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Python_(programming_language)"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# Title
title = soup.find("h1").get_text(strip=True)
print("Title:", title)

# First meaningful paragraph
content = soup.find("div", class_="mw-parser-output")

first_paragraph = None
for p in content.find_all("p"):
    text = p.get_text(" ", strip=True)
    if text:
        first_paragraph = text
        break

print("\nFirst paragraph:\n", first_paragraph)

Title: Python (programming language)

First paragraph:
 Python is a high-level , general-purpose programming language . Its design philosophy emphasizes code readability with the use of significant indentation . [ 38 ] Python is dynamically type-checked and garbage-collected . It supports multiple programming paradigms , including structured (particularly procedural ), object-oriented and functional programming .


#### All section headings

In [7]:
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Python_(programming_language)"
response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
soup = BeautifulSoup(response.text, "html.parser")

headings = soup.find_all(["h2", "h3"])

for h in headings:
    text = h.get_text(strip=True)
    if text:
        print(text)

Contents
History
Design philosophy and features
Syntax and semantics
Indentation
Statements and control flow
Expressions
Typing
Arithmetic operations
Function syntax
Code examples
Libraries
Development environments
Implementations
Reference implementation
Limitations of the reference implementation
Other implementations
Unsupported implementations
Transpilers to other languages
Performance
Language development
Naming
Languages influenced by Python
See also
Notes
References
Sources
Further reading
External links


#### Links from the article

In [9]:
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Python_(programming_language)"
response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
soup = BeautifulSoup(response.text, "html.parser")

for link in soup.select("a[href]"):
    href = link["href"]
    if href.startswith("/wiki/") and ":" not in href:
        print("https://en.wikipedia.org" + href)

https://en.wikipedia.org/wiki/Main_Page
https://en.wikipedia.org/wiki/Main_Page
https://en.wikipedia.org/wiki/Python_(programming_language)
https://en.wikipedia.org/wiki/Python_(programming_language)
https://en.wikipedia.org/wiki/Python_(programming_language)
https://en.wikipedia.org/wiki/Programming_paradigm
https://en.wikipedia.org/wiki/Multi-paradigm
https://en.wikipedia.org/wiki/Object-oriented_programming
https://en.wikipedia.org/wiki/Procedural_programming
https://en.wikipedia.org/wiki/Imperative_programming
https://en.wikipedia.org/wiki/Functional_programming
https://en.wikipedia.org/wiki/Structured_programming
https://en.wikipedia.org/wiki/Reflective_programming
https://en.wikipedia.org/wiki/Software_design
https://en.wikipedia.org/wiki/Guido_van_Rossum
https://en.wikipedia.org/wiki/Software_developer
https://en.wikipedia.org/wiki/Python_Software_Foundation
https://en.wikipedia.org/wiki/Software_release_life_cycle
https://en.wikipedia.org/wiki/Type_system
https://en.wikipedia.o

#### Autoamted

In [12]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

HEADERS = {"User-Agent": "Mozilla/5.0"}

PAGES = [
    "Python_(programming_language)",
    "Artificial_intelligence",
    "Data_science",
    "Sociology"
]

def make_session():
    session = requests.Session()
    retry = Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session

def scrape_wikipedia_page(session, page_name):
    url = f"https://en.wikipedia.org/wiki/{page_name}"

    try:
        response = session.get(url, headers=HEADERS, timeout=20)
        response.raise_for_status()
    except requests.RequestException as e:
        return {
            "page": page_name,
            "url": url,
            "title": None,
            "first_paragraph": None,
            "status": f"error: {e}"
        }

    soup = BeautifulSoup(response.text, "html.parser")
    title = soup.find("h1").get_text(" ", strip=True) if soup.find("h1") else None

    content = soup.find("div", class_="mw-parser-output")
    first_paragraph = None

    if content:
        for p in content.find_all("p"):
            text = p.get_text(" ", strip=True)
            if text and len(text) > 50:
                first_paragraph = text
                break

    return {
        "page": page_name,
        "url": url,
        "title": title,
        "first_paragraph": first_paragraph,
        "status": "ok"
    }

session = make_session()
results = []

for page in PAGES:
    print(f"Processing {page}...")
    results.append(scrape_wikipedia_page(session, page))
    time.sleep(1)

df = pd.DataFrame(results)
df.to_excel("wiki_automated_scrape.xlsx", index=False)
print(df)

Processing Python_(programming_language)...
Processing Artificial_intelligence...
Processing Data_science...
Processing Sociology...
                            page  \
0  Python_(programming_language)   
1        Artificial_intelligence   
2                   Data_science   
3                      Sociology   

                                                 url  \
0  https://en.wikipedia.org/wiki/Python_(programm...   
1  https://en.wikipedia.org/wiki/Artificial_intel...   
2         https://en.wikipedia.org/wiki/Data_science   
3            https://en.wikipedia.org/wiki/Sociology   

                           title  \
0  Python (programming language)   
1        Artificial intelligence   
2                   Data science   
3                      Sociology   

                                     first_paragraph status  
0  Python is a high-level , general-purpose progr...     ok  
1                                               None     ok  
2  Data science is an interdisciplinar

### Bank of local data

In [14]:
import requests
import pandas as pd

url = "https://bdl.stat.gov.pl/api/v1/data/by-variable/3643?format=json&year=2015&year=2020"

response = requests.get(url)
data = response.json()

print(data.keys())

dict_keys(['totalRecords', 'links', 'variableId', 'measureUnitId', 'aggregateId', 'lastUpdate', 'results'])


In [15]:
records = []

for item in data['results']:
    unit = item['name']
    for val in item['values']:
        records.append({
            "region": unit,
            "year": val['year'],
            "value": val['val']
        })

df = pd.DataFrame(records)
print(df.head())

                   region  year  value
0                  POLSKA  2015     68
1                  POLSKA  2020     53
2  MAKROREGION POŁUDNIOWY  2015     11
3  MAKROREGION POŁUDNIOWY  2020      7
4             MAŁOPOLSKIE  2015      6


In [ ]:
Client registered: e34e20cd-163d-4b02-082e-08de83faec90

In [ ]:
import requests
import pandas as pd

API_KEY = None  # put your key here if you have one
HEADERS = {"X-ClientId": API_KEY} if API_KEY else {}

# Example pattern only: replace VARIABLE_ID with the unemployment variable you choose
url = "https://bdl.stat.gov.pl/api/v1/data/by-variable/VARIABLE_ID?format=json"

response = requests.get(url, headers=HEADERS, timeout=30)
response.raise_for_status()
data = response.json()

records = []
for item in data.get("results", []):
    unit_name = item.get("name")
    unit_id = item.get("id")
    for v in item.get("values", []):
        records.append({
            "unit_id": unit_id,
            "unit_name": unit_name,
            "year": v.get("year"),
            "value": v.get("val")
        })

df = pd.DataFrame(records)
print(df.head())

In [1]:
import requests
from pathlib import Path

def download_gutenberg_book(book_id: int, out_dir: str = "books") -> str:
    """
    Download a Project Gutenberg plain-text book by ID.
    Tries common text file patterns used by Gutenberg.
    Returns the local file path.
    """
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    # Common plain-text URL patterns
    candidate_urls = [
        f"https://www.gutenberg.org/files/{book_id}/{book_id}-0.txt",
        f"https://www.gutenberg.org/files/{book_id}/{book_id}.txt",
        f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt",
    ]

    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; educational-use script)"
    }

    for url in candidate_urls:
        r = requests.get(url, headers=headers, timeout=30)
        if r.status_code == 200 and len(r.text) > 1000:
            out_path = Path(out_dir) / f"gutenberg_{book_id}.txt"
            out_path.write_text(r.text, encoding="utf-8")
            return str(out_path)

    raise ValueError(f"Could not download book ID {book_id} in plain text.")

# Example: Pride and Prejudice = 1342
path = download_gutenberg_book(1342)
print("Saved to:", path)

Saved to: books/gutenberg_1342.txt


In [1]:
with open("books/gutenberg_1342.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(text[:2000])

*** START OF THE PROJECT GUTENBERG EBOOK 1342 ***




                            [Illustration:

                             GEORGE ALLEN
                               PUBLISHER

                        156 CHARING CROSS ROAD
                                LONDON

                             RUSKIN HOUSE
                                   ]

                            [Illustration:

               _Reading Jane’s Letters._      _Chap 34._
                                   ]




                                PRIDE.
                                  and
                               PREJUDICE

                                  by
                             Jane Austen,

                           with a Preface by
                           George Saintsbury
                                  and
                           Illustrations by
                             Hugh Thomson

                         [Illustration: 1894]

                       Ruskin       156. Charing

## EXCERSISES
- Choose link from Wikipedia to be scrapped


## HOMEWORK

- https://www.jmlr.org/papers/volume3/blei03a/blei03a.pdf

In [ ]:
import scipy

scipy.io.wavfile.write("techno.wav", rate=model.config.sampling_rate, data=output)
